In [1]:
import polars as pl
import polars.selectors as cs
import plotly.graph_objects as go
import plotly.express as px
from typing import Union, Optional

In [2]:
df = pl.read_parquet('../data/raw/tripdata.parquet')

Convert timestamps from String to Datetime

In [3]:
df = df.with_columns(
    pl.col("started_at").str.to_datetime(),
    pl.col("ended_at").str.to_datetime(),
)

In [4]:
height_before = df.height

df = df.unique()

print(f"{height_before - df.height} duplicate records were excluded")

35 duplicate records were excluded


In [5]:
height_before = df.height

df = df.filter(
    pl.col("ended_at")
    .sub(pl.col("started_at"))
    .dt.total_seconds() >= 60
)

print(f"{height_before - df.height} rides less than 1 minute long were excluded")

159221 rides less than 1 minute long were excluded


In [6]:
height_before = df.height

df = df.filter(
    pl.col("started_at") < pl.col("ended_at")
)

print(f"{height_before - df.height} rides that start before they end were excluded")

0 rides that start before they end were excluded


Since records missing end_lng are also mission end station name and ID, then we cannot determine end station info. Therefore, we exclude those records.

In [7]:
height_before = df.height

df = df.drop_nulls(subset=["end_lng"])

print(f"{height_before - df.height} records without any end station info were excluded")

5445 records without any end station info were excluded


In [7]:
df = df.with_columns(
    pl.col("end_s")
)

In [8]:
df.glimpse()

Rows: 6007084
Columns: 13
$ ride_id                     <str> '62313B3B6BB5F3EA', '653EF414D2F3E86F', '9C688079281D6B9E', 'F8DF2F87E3CC4535', '4091B21626D094DF', '6DB8703B83D47BF3', '3EEA0BD826A82789', 'E277CB89F8080AE6', '1B1C2604AEB0AE0D', '65EDACF8E096CAB7'
$ rideable_type               <str> 'classic_bike', 'classic_bike', 'electric_bike', 'electric_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'classic_bike', 'classic_bike', 'classic_bike'
$ started_at         <datetime[μs]> 2026-07-25 11:11:21.672000, 2026-05-17 19:38:58.422000, 2026-07-12 17:23:22.069000, 2026-06-19 21:40:37.768000, 2026-08-10 21:29:28.936000, 2026-07-13 15:31:35.579000, 2026-04-16 13:09:33.485000, 2026-08-15 14:09:11.369000, 2026-08-16 15:44:33.643000, 2026-05-26 18:51:28.793000
$ ended_at           <datetime[μs]> 2026-07-25 11:42:54.896000, 2026-05-17 19:54:46.305000, 2026-07-12 17:54:20.280000, 2026-06-19 21:47:35.904000, 2026-08-10 21:49:55.380000, 2026-07-13 15:36:54.934000, 2026-04-16 13:29:20.02